## Setup

In [ ]:
# Install those librairies first
# This command should be used in a terminal :
!pip install xarray "zarr>=3.1.6" imagecodecs fsspec requests aiohttp matplotlib

# Extra dependencies for the streaming download / export blocks 
!pip install "dask[array]" rasterio tqdm

## Initialization

In [ ]:
# Imports
import os
import xarray as xr
from imagecodecs.zarr import register_codecs

# Register image codecs for Zarr
register_codecs()

In [ ]:
URL="https://stac.webservice-energy.org/data/allskyimager/ikarus/"

## Load metadata

In [ ]:
# Access remote data : only metadata is loaded at this stage
ds = xr.open_dataset(URL, engine="zarr")

# Overview of available variables and dimensions
ds

## Load the data

You can load a subset of data for a speicifc variable, from the remote host.


In [ ]:
# Load all HDR images for a single day
images = ds.hdr_image.sel(time="2024-08-15").load()

# The result is a array Dataarray : a 4D cube of data with proper labels on the axes
images.dims

## Animate

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Show an animation
fig, ax = plt.subplots()
im = ax.imshow(images.isel(time=35).values)
ax.axis("off")

def update(t):
    im.set_data(images.isel(time=t))
    ax.set_title(f"Frame {t}")
    return [im]

anim = FuncAnimation(
    fig, update,
    frames=range(35, images.sizes["time"]),
    interval=100, blit=True)

plt.close(fig)
HTML(anim.to_jshtml())

# Download zarr locally 

To download an offline Zip of a subset, you may use [this web form](https://stac.webservice-energy.org/zarr-subset/?url=/data/allskyimager/ikarus/)


## Export images 

The following code allows you to export the dataset as images (Tiff/Jpeg) or disk.

In [ ]:
import os
import warnings
import numpy as np
import xarray as xr
import rasterio
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

def _image_vars(ds):
    """Temporal image variables: name -> (x_dim, y_dim, channel_dim)."""
    found = {}
    for name, da in ds.data_vars.items():
        if "time" not in da.dims:
            continue
        xd = next((d for d in da.dims if d.endswith("_x")), None)
        yd = next((d for d in da.dims if d.endswith("_y")), None)
        cd = next((d for d in da.dims if d in ("rgb", "band")), None)
        if xd and yd and cd:
            found[name] = (xd, yd, cd)
    return found

COMPRESSION_PRESETS = {
    "lerc": dict(COMPRESS="LERC_ZSTD", ZSTD_LEVEL=9, MAX_Z_ERROR=0),
    "zstd": dict(COMPRESS="ZSTD", ZSTD_LEVEL=9, PREDICTOR=3),
    "lzw": dict(COMPRESS="LZW", PREDICTOR=2),
    "deflate": dict(COMPRESS="DEFLATE", PREDICTOR=2),
}


def _variable_config(options):
    """Translate the compact per-variable config into (ext, rasterio options)."""
    fmt = str(options.get("format") or "tif").lower()
    if fmt == "jpg":
        return ".jpg", dict(driver="JPEG", QUALITY=int(options.get("quality", 95)))
    if fmt != "tif":
        raise ValueError(f"unknown format '{options.get('format')}' (use 'tif' or 'jpg')")
    comp = options.get("compression")
    if comp is None or str(comp).lower() in ("none", ""):
        return ".tif", dict(driver="GTiff", TILED="YES")
    comp = str(comp).lower()
    if comp not in COMPRESSION_PRESETS:
        raise ValueError(f"unknown compression '{options.get('compression')}' "
                         f"(use {', '.join(COMPRESSION_PRESETS)} or None)")
    return ".tif", dict(driver="GTiff", TILED="YES") | COMPRESSION_PRESETS[comp]


def _profile_and_ext(bands, options):
    """Raster profile = frame geometry (height/width/count/dtype) merged with
    the per-variable options derived from the compact config."""
    header = dict(height=bands.shape[1], width=bands.shape[2],
                  count=bands.shape[0], dtype=bands.dtype.name)
    ext, copts = _variable_config(options)
    return header | copts, ext


def _set_colorinterp(dst):
    from rasterio.enums import ColorInterp
    if dst.driver != "GTiff" or dst.count < 3:
        return
    ci = list(dst.colorinterp)
    for i, c in enumerate((ColorInterp.red, ColorInterp.green, ColorInterp.blue)):
        ci[i] = c
    dst.colorinterp = ci


def _shard_chunks(da):
    """Chunk dict aligned on the store shards (-1 means full-dimension)."""
    try:
        return {d: (n if n > 0 else da.sizes[d]) for d, n in da.zarr_shards.items()}
    except Exception:
        return {d: da.sizes[d] for d in da.dims}


def _time_range_slice(time_range):
    """Normalize the optional temporal slice to a ``slice`` object.

    Accepted: None (full extent), a ``slice``, a (start, stop) pair, or a
    single start timestamp (open-ended). Labels are anything ``sel`` accepts
    (ISO datetime strings, np.datetime64, ...).
    """
    if time_range is None:
        return slice(None)
    if isinstance(time_range, slice):
        return time_range
    if isinstance(time_range, (tuple, list)):
        if len(time_range) == 2:
            return slice(*time_range)
        if len(time_range) == 1:
            return slice(time_range[0], None)
        raise ValueError("time_range must be a slice or a (start, stop) pair")
    return slice(time_range, None)


def _write_block(block, ci, ctx):
    """Compress one shard-aligned dask block to one file per frame.

    The delivered block keeps every array dimension (chunk shapes aligned on
    the store shards); ``ci`` is the block's grid position per dimension.
    ``ctx`` carries the per-variable geometry and profile to avoid any
    late-binding closure on loop variables.
    """
    oth = ctx["oth"]
    starts, labels = ctx["starts"], ctx["labels"]
    dims = ctx["dims"]
    for offs in np.ndindex(*(block.shape[dims[d]] for d in oth)):
        sl = dict(zip(oth, offs))
        slot = {d: starts[d][ci[d]] + sl[d] for d in oth}
        t = np.datetime64(labels["time"][slot["time"]]).astype("datetime64[s]").item()
        date, fname = t.strftime("%Y-%m-%d"), t.strftime("%H%M%S")
        idx = [slice(None)] * block.ndim
        for d in oth:
            idx[dims[d]] = sl[d]
        bands = block[tuple(idx)]
        sub = [str(labels[d][slot[d]]) for d in oth if d != "time"]
        outdir = os.path.join(ctx["dest"], ctx["name"], date, *sub)
        os.makedirs(outdir, exist_ok=True)
        with rasterio.open(os.path.join(outdir, fname + ctx["ext"]), "w", **ctx["profile"]) as d:
            d.write(bands)
            _set_colorinterp(d)


def export_images(
    source, 
    variables=None, 
    dest="./output/images/",
    time_range=None, 
    workers=4) :
    """Write the selected image variables to rasters, parallelized as
    shard-aligned dask blocks (each shard decoded once, compressed in parallel).

    ``source`` is a zarr path/URL (remote store or local download).
    ``variables`` selects the variables to export and their compression as a
    compact ``{name: {format, compression, quality}}`` config (see
    DEFAULT_VARIABLES). Defaults to DEFAULT_VARIABLES; unknown names are
    skipped with a warning.
    ``time_range`` is an optional temporal slice: None (full extent), a
    ``slice``, a (start, stop) pair, or a single start timestamp.
    Layout: ``<dest>/<var>/<date>/<other-dim...>/<time>.jpg|.tif``.
    Uses the dask threaded scheduler: rasterio/numpy release the GIL, and it is
    safe to run from a notebook (the process scheduler's spawn re-imports the
    kernel main module and crashes).
    """
    import dask
    from dask.diagnostics import ProgressBar

    meta = xr.open_zarr(source, consolidated=False)
    image_vars = _image_vars(meta)
    tr = _time_range_slice(time_range)
    if tr != slice(None):
        meta = meta.sel(time=tr)
    variables = variables or {n: o for n, o in DEFAULT_VARIABLES.items() if n in image_vars}
    unknown = [n for n in variables if n not in image_vars]
    if unknown:
        print(f"skipping missing/non-image variable(s): {unknown}")

    tasks = []
    for name, options in variables.items():
        if name not in image_vars:
            continue
        xd, yd, cd = image_vars[name]
        da = xr.open_zarr(source, chunks=_shard_chunks(meta[name]),
                          consolidated=False)[name]
        if tr != slice(None):
            da = da.sel(time=tr)
        da = da.transpose(..., cd, yd, xd)
        oth = [d for d in da.dims if d not in (cd, yd, xd)]
        labels = {d: da[d].values for d in oth}
        dims = {d: i for i, d in enumerate(da.dims)}
        chunks = da.chunks
        starts = {d: np.concatenate([[0], np.cumsum(chunks[dims[d]])[:-1]]) for d in oth}
        sample = np.empty((da.sizes[cd], da.sizes[yd], da.sizes[xd]), dtype=da.dtype)
        profile, ext = _profile_and_ext(sample, options)
        ctx = dict(dest=dest, name=name, ext=ext, profile=profile,
                   oth=oth, dims=dims, starts=starts, labels=labels)
        blocks = da.data.to_delayed()
        for bi in np.ndindex(blocks.shape):
            ci = {d: (bi[dims[d]] if len(chunks[dims[d]]) > 1 else 0) for d in da.dims}
            tasks.append(dask.delayed(_write_block)(blocks[bi], ci, ctx))

    print(f"{len(tasks)} shard-aligned block(s), workers={workers}")
    with dask.config.set(scheduler="threads", num_workers=workers):
        with ProgressBar():
            dask.compute(*tasks)

    print(f"Saved in {dest}")


In [ ]:
# Export data as images on disc
export_images(
    URL,

    # Variables and associated formats
    variables={
        "raw_sensor": dict(format="tif", compression="LERC"), # Good for int format
        "radiance": dict(format="tif", compression="ZSTD"),
        "hdr_image": dict(format="jpg"),
    }, 
    dest="./output/images/",

    # Optional time range
    time_range=slice("2024", "2025"),
    workers=4)